# Laboratorio 07 — Mudarse sin cerrar el negocio

**Duración: 60 minutos.**

Este es **el laboratorio que van a repetir en su plataforma**, con sus propias tablas,
en los próximos meses. Todo lo anterior del curso fue preparación para hoy.

El laboratorio pasado cerró con esto:

> Todo lo que hiciste fue sobre tablas Iceberg que **creaste tú, desde cero**. Pero la
> DGT no parte de cero: tiene años de documentos en **tablas Hive**, en producción, de
> las que dependen los reportes de todos los meses.
>
> **¿Cómo se pasan a Iceberg?**

---

### Migrar no es copiar y rezar

Una migración tiene tres momentos, y el alumno que se salta uno se lleva un susto:

1. **Preparar** — saber exactamente qué se tiene. Contar antes.
2. **Convertir** — el paso que todo el mundo cree que es la migración entera.
3. **Verificar** — comprobar, número contra número, que no se perdió nada.

**El tercero es el que la gente omite**, y es el que separa una migración profesional
de un susto. Hoy vas a hacer los tres, en ese orden.

### En el almacén

Mudarse de local **sin cerrar ni un día**, y con la certeza de que no se perdió ni un
producto en el camino. El inventario de antes y el de después tienen que dar
exactamente lo mismo; si no, no fue una mudanza, fue una pérdida.

---

### Cómo se usa este notebook

Como siempre. Tres cosas de hoy:

- Vas a empezar creando **una tabla Hive de verdad**, no Iceberg. Es el punto de
  partida: la tabla que la DGT tiene hoy.
- Hay **dos celdas que fallan a propósito**, y van anunciadas.
- El cuaderno termina en el `SHOW TABLES`, pero el laboratorio no: el último tramo, el
  paso 6, lo haces **en Hue**, en el navegador, sobre la tabla que acabas de migrar. Es
  por donde se conectan las herramientas de reportería de la institución.


## Paso 0 — La tabla Hive de partida

Primero, a tu espacio de trabajo.

In [1]:
%%sql
-- Celda 0.1
USE mi_espacio

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/24 21:20:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Partimos de cero.

In [2]:
%%sql
-- Celda 0.2
DROP TABLE IF EXISTS documentos_hive

Y también el respaldo, si quedó de una corrida anterior. Al final del laboratorio
vas a entender por qué existe ese nombre.

In [3]:
%%sql
-- Celda 0.3
DROP TABLE IF EXISTS documentos_hive_backup_

Ahora la tabla de partida, y fíjate en cómo se declara: **no dice `USING
iceberg`**. Dice `STORED AS PARQUET`, que es la forma clásica de crear una tabla Hive.

Esta es, exactamente, la clase de tabla que la DGT tiene hoy en su data lake.

In [4]:
%%sql
-- Celda 0.4
CREATE TABLE documentos_hive (
    rut_emisor     STRING,
    tipo_dte       INT,
    folio          BIGINT,
    fecha_emision  DATE,
    monto_neto     DECIMAL(18,2),
    monto_iva      DECIMAL(18,2),
    monto_total    DECIMAL(18,2),
    estado_sii     STRING
) STORED AS PARQUET

Y la cargamos con un año de documentos.

In [5]:
%%sql
-- Celda 0.5
INSERT INTO documentos_hive
SELECT rut_emisor, tipo_dte, folio, fecha_emision,
       monto_neto, monto_iva, monto_total, estado_sii
FROM curso.dte_2024

Comprobemos qué clase de tabla es. Busca la línea **`Provider`** en la salida.

In [6]:
%%sql
-- Celda 0.6
DESCRIBE TABLE EXTENDED documentos_hive

col_name,data_type,comment
rut_emisor,string,None
tipo_dte,int,None
folio,bigint,None
fecha_emision,date,None
monto_neto,"decimal(18,2)",None
monto_iva,"decimal(18,2)",None
monto_total,"decimal(18,2)",None
estado_sii,string,None
,,
# Detailed Table Information,,


## Paso 1 — Levantar el inventario

`Provider` dice **hive**. Y más abajo verás `Serde Library` con
`ParquetHiveSerDe`: es una tabla Hive corriente, leída como Hive.

---

Ahora, **antes de tocar nada**, el inventario.

Esto no es trámite: una migración se verifica **contra lo que había**, y lo que había
solo se sabe si se midió antes. Si migras primero y cuentas después, no estás
verificando nada: estás mirando un número solo y esperando que esté bien.

Primera medición: cuántos documentos y por cuánto.

In [7]:
%%sql
-- Celda 1.1
SELECT count(*)         AS documentos,
       sum(monto_total) AS total_general
FROM documentos_hive

documentos,total_general
30000,291293351462.71


**Anota esos dos números.** Los vas a comparar en veinte minutos.

Segunda medición, más fina: el desglose por período. Si algo se pierde en la mudanza,
aquí se ve **en qué mes** se perdió.

In [8]:
%%sql
-- Celda 1.2
SELECT date_format(fecha_emision, 'yyyy-MM') AS periodo,
       count(*)                             AS documentos,
       sum(monto_total)                     AS total_del_periodo
FROM documentos_hive
GROUP BY date_format(fecha_emision, 'yyyy-MM')
ORDER BY periodo

periodo,documentos,total_del_periodo
2024-01,2500,23381729910.07
2024-02,2500,24831948613.59
2024-03,2500,26699931432.64
2024-04,2500,23722997990.55
2024-05,2500,24328007216.84
2024-06,2500,24547341762.59
2024-07,2500,23621609483.83
2024-08,2500,23648615665.89
2024-09,2500,24136964210.51
2024-10,2500,23106058024.78


## Paso 2 — Mirar sus límites

Doce períodos, dos mil quinientos documentos cada uno. Ese es el inventario.

Antes de mudarnos, veamos por qué vale la pena. Intenta sobre esta tabla algo que ya
sabes hacer desde el laboratorio 01: pedirle su historial.

> **Esta celda va a fallar.** Está puesta a propósito.

In [9]:
%%sql
-- Celda 2.1
SELECT snapshot_id, committed_at, operation
FROM mi_espacio.documentos_hive.snapshots
ORDER BY committed_at

No hay tal cosa. Una tabla Hive **no tiene páginas**: tiene el estado de ahora y
nada más.

Probemos lo otro: deshacer una carga, como en el laboratorio 01. El número que va al
final da lo mismo, ya vas a ver por qué.

> **Esta también falla, y es la última.**

In [10]:
%%sql
-- Celda 2.2
CALL spark_catalog.system.rollback_to_snapshot('mi_espacio.documentos_hive', 1)

## Paso 3 — Migrar

`is not org.apache.iceberg.spark.source.SparkTable`: **ni siquiera llegó a mirar el
número**. La tabla no es Iceberg, así que no hay nada que revertir.

Ahí está, en una línea, todo lo que la DGT no tiene hoy: sin historial, sin reversión,
sin poder responder qué decía un dato antes.

---

Mudémonos. `system.migrate` convierte la tabla **en el lugar**, y conviene saber
exactamente qué hace antes de ejecutarlo:

| Qué hace | Detalle |
|---|---|
| **No copia los datos** | Los archivos Parquet que ya existen se reutilizan tal cual |
| **Escribe metadatos nuevos** | Crea los manifiestos y la primera página de la tabla Iceberg |
| **Conserva el nombre** | La tabla se sigue llamando igual, y las consultas de siempre siguen funcionando |
| **Guarda un respaldo** | La definición Hive original queda como `documentos_hive_backup_` |

Que no copie los datos es lo que hace viable migrar años de historia: sobre una tabla
de doscientos millones de documentos, esto demora lo mismo que sobre la tuya.

In [11]:
%%sql
-- Celda 3.1
CALL spark_catalog.system.migrate('mi_espacio.documentos_hive')

migrated_files_count
1


`migrated_files_count` dice cuántos archivos de datos quedaron incorporados.

Veamos qué clase de tabla es ahora. Misma consulta del paso 0, misma línea que hay que
mirar.

In [12]:
%%sql
-- Celda 3.2
DESCRIBE TABLE EXTENDED documentos_hive

col_name,data_type,comment
rut_emisor,string,
tipo_dte,int,
folio,bigint,
fecha_emision,date,
monto_neto,"decimal(18,2)",
monto_iva,"decimal(18,2)",
monto_total,"decimal(18,2)",
estado_sii,string,
,,
# Partitioning,,


## Paso 4 — Verificar

**`Provider` ahora dice `iceberg`.** Misma tabla, mismo nombre, otra naturaleza.

Y aquí viene el paso que la gente omite. **Verificar.**

Repite la primera medición del paso 1, **exactamente la misma consulta**, y compárala
con lo que anotaste.

In [13]:
%%sql
-- Celda 4.1
SELECT count(*)         AS documentos,
       sum(monto_total) AS total_general
FROM documentos_hive

documentos,total_general
30000,291293351462.71


Compáralo con el paso 1: **el mismo conteo y el mismo total, al centavo.**

Ahora la segunda medición.

In [14]:
%%sql
-- Celda 4.2
SELECT date_format(fecha_emision, 'yyyy-MM') AS periodo,
       count(*)                             AS documentos,
       sum(monto_total)                     AS total_del_periodo
FROM documentos_hive
GROUP BY date_format(fecha_emision, 'yyyy-MM')
ORDER BY periodo

periodo,documentos,total_del_periodo
2024-01,2500,23381729910.07
2024-02,2500,24831948613.59
2024-03,2500,26699931432.64
2024-04,2500,23722997990.55
2024-05,2500,24328007216.84
2024-06,2500,24547341762.59
2024-07,2500,23621609483.83
2024-08,2500,23648615665.89
2024-09,2500,24136964210.51
2024-10,2500,23106058024.78


## Paso 5 — Comprobar que ahora sí es Iceberg

Doce períodos, los mismos documentos, los mismos totales. **La mudanza no perdió
nada**, y no lo estás suponiendo: lo comprobaste.

Si alguno de estos números no hubiera coincidido, la migración no estaría terminada, y
lo sabrías **ahora** y no cuando alguien reclame por un reporte.

---

Y ahora lo que en el paso 2 no se pudo. La misma consulta que falló hace cinco
celdas.

In [15]:
%%sql
-- Celda 5.1
SELECT snapshot_id, committed_at, operation
FROM mi_espacio.documentos_hive.snapshots
ORDER BY committed_at

snapshot_id,committed_at,operation
8310466069811688717,2026-09-24 21:21:00.495,append


**Una página.** La tabla acaba de nacer a la vida de Iceberg y ya tiene memoria: de
aquí en adelante cada carga, cada corrección y cada borrado van a dejar la suya.

Miremos también sus archivos, como en el laboratorio 02.

In [16]:
%%sql
-- Celda 5.2
SELECT count(*)          AS archivos,
       sum(record_count) AS documentos
FROM mi_espacio.documentos_hive.files

archivos,documentos
1,30000


Los mismos archivos Parquet de siempre: **no se copió nada**, solo se les puso un
índice encima.

Última cosa antes de salir del notebook. ¿Qué tablas hay ahora en tu espacio?

In [17]:
%%sql
-- Celda 5.3
SHOW TABLES IN mi_espacio

namespace,tableName,isTemporary
mi_espacio,contribuyentes_lab00,False
mi_espacio,contribuyentes_lab01,False
mi_espacio,contribuyentes_lab02,False
mi_espacio,contribuyentes_lab09,False
mi_espacio,contribuyentes_lab14,False
mi_espacio,contribuyentes_lab15,False
mi_espacio,contribuyentes_lab16,False
mi_espacio,documentos_hive,False
mi_espacio,documentos_hive_backup_,False
mi_espacio,documentos_lab03,False


## Paso 6 — La capa de consulta

Esto **no va en el cuaderno**. Se escribe en Hue, en el navegador: abre otra
pestaña, entra con tu usuario y abre el editor **SparkSQL (Iceberg)**. Son dos consultas
de reportería sobre la tabla que acabas de migrar, y en Hue el nombre de la tabla va
**completo**, con tu espacio adelante.

**Se escribe** (en Hue).

```sql-hue
SELECT date_format(fecha_emision, 'yyyy-MM') AS periodo,
       count(*)                              AS documentos,
       sum(monto_total)                      AS total_del_periodo
FROM mi_espacio.documentos_hive
GROUP BY date_format(fecha_emision, 'yyyy-MM')
ORDER BY periodo
```

**En consola** doce filas, **exactamente las mismas del paso 4**.

```
| periodo | documentos | total_del_periodo |
| 2024-01 | 2500       | 23381729910.07    |
| 2024-02 | 2500       | 24831948613.59    |
| 2024-03 | 2500       | 26699931432.64    |
…
(12 filas)
```

**Varía entre alumnos** tu database en la consulta.

---

**Se escribe** (en Hue).

```sql-hue
SELECT tipo_dte,
       count(*)         AS documentos,
       sum(monto_total) AS total
FROM mi_espacio.documentos_hive
GROUP BY tipo_dte
ORDER BY tipo_dte
```

**En consola**

```
| tipo_dte | documentos | total             |
| 33       | 12664      | 241123556352.49   |
| 34       | 2138       | 32447276599.00    |
| 39       | 11365      | 817555482.66      |
| 52       | 1766       | 30041653364.00    |
| 56       | 597        | 8524913042.87     |
| 61       | 1470       | -21661603378.31   |
(6 filas)
```

**Varía entre alumnos** tu database en la consulta.

---

## Antes de irte: el respaldo

Ahí está `documentos_hive_backup_`, con la definición Hive original. Y aquí va **la
advertencia más importante del laboratorio**, porque es la que puede costar caro en
producción:

> **Los archivos de datos quedaron bajo el directorio del respaldo.**
>
> La tabla Iceberg apunta a ellos, pero físicamente viven en la carpeta de
> `documentos_hive_backup_`. Si alguien borra esa tabla con eliminación de datos, **se lleva
> los datos de la tabla migrada**.
>
> El respaldo se conserva mientras dure la validación de la migración. Cuando se
> decida eliminarlo, se hace **con conocimiento de causa** y comprobando antes dónde
> están los archivos, con la consulta de `.files` que acabas de usar.

---

## Listo

### Lo que te llevas

Una migración tiene tres momentos, y hoy hiciste los tres:

| Momento | Lo que hiciste |
|---|---|
| **Preparar** | Contaste antes: total, monto, desglose por período |
| **Convertir** | `CALL spark_catalog.system.migrate('<db>.<tabla>')` |
| **Verificar** | Repetiste las mismas mediciones y comparaste número contra número |

Y lo que la tabla ganó en el camino: historial, reversión, viaje en el tiempo,
evolución de esquema y de particiones, ramas. Todo lo del curso, sobre datos que ya
estaban ahí.

### Los otros caminos, y cuándo usar cada uno

`migrate` no es el único. Los cuatro funcionan en esta plataforma y sirven para cosas
distintas:

| Camino | Qué hace | Cuándo |
|---|---|---|
| `system.migrate` | Convierte **en el lugar**, conserva el nombre, deja respaldo | La migración de verdad, cuando ya decidiste pasarte |
| `system.snapshot` | Crea una tabla Iceberg **nueva** sobre los mismos archivos, sin tocar la original | **Para probar sin riesgo.** La tabla Hive sigue en producción intacta |
| `CREATE TABLE … USING iceberg AS SELECT …` | **Copia** los datos a una tabla nueva | Cuando además quieres cambiar el esquema, particionar distinto o filtrar |
| `system.add_files` | Suma archivos existentes a una tabla Iceberg ya creada | Cargas incrementales desde un directorio que ya existe |

**El orden recomendado para la DGT**: primero `system.snapshot` sobre una tabla de
prueba, para validar sin tocar producción; después `system.migrate` sobre la tabla de
verdad, con el inventario levantado antes.

### Una advertencia para cuando lo hagan en su plataforma

Este ambiente usa el mismo catálogo que su CDP, pero **no tiene Kerberos ni Ranger**.
`migrate` renombra la tabla original, y renombrar exige permisos sobre el catálogo que
en su plataforma pueden no estar dados. Si `migrate` falla por permisos, el camino de
`system.snapshot` no renombra nada y suele pasar donde el otro no.

Y una segunda: si hay trabajos programados que escriben en la tabla mientras se migra,
párenlos. `migrate` no cierra el negocio, pero tampoco atiende a dos cajeros a la vez.

### La pregunta del laboratorio siguiente

Tu tabla ya está migrada y desde mañana recibe documentos todos los días.

Con el tiempo van a pasar dos cosas, y las dos las viste venir en laboratorios
anteriores: **se va a llenar de archivos pequeños** —¿te acuerdas de los 31 archivos
para un solo mes del laboratorio 04?— y **va a acumular páginas viejas** que ya nadie
consulta.

**¿Cómo se mantiene una tabla que trabaja todos los días?** Eso es el **Laboratorio 08
— Dejar la libreta ordenada**, el último del curso.
